# 03 — Causal Inference: Does GDP Causally Affect Life Expectancy?

**Research question:** Does GDP *causally* affect life expectancy? Through what mechanisms? With what time lags? Does the relationship vary by income level?

This notebook runs five complementary causal inference methods on a 29-country × 25-year panel (2000–2024):

| Method | Identification strategy | Handles |
|--------|------------------------|---------||
| Granger causality | Time-precedence test | Dynamic, bidirectional causality |
| Panel FE (TWFE) | Within-country variation | Unobserved country heterogeneity |
| IV-2SLS | External demand instruments | Endogeneity (reverse causality) |
| Diff-in-Diff | Policy shocks (UHC reforms) | Confounders via parallel trends |
| Synthetic control | Counterfactual construction | Country-specific treatment effects |

**Primary variable:** life expectancy (years) 
**Key regressor:** log GDP per capita, PPP (World Bank WDI) 
**Controls:** health expenditure, education expenditure, governance (WGI), urbanisation, fertility, trade openness, age structure, inflation

In [ ]:
import sys, os, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

from src.analysis.causal import run_all_causal, load_df
from src.visualization.causal_plots import run_all_causal_plots, CAUSAL_FIG_DIR
from src.analysis.tables import run_all_tables

df = load_df()
print(f'Dataset: {df.shape[0]} obs, {df["iso3"].nunique()} countries, {df["year"].min()}-{df["year"].max()}')

## Run all causal methods
This single call runs all five methods, subgroup analysis, and robustness checks. Runtime ~2-3 minutes.

In [ ]:
results = run_all_causal(df)

---
## 1. Granger Causality

**Method:** For each of the 29 countries, we test whether past values of log GDP improve forecasts of life expectancy beyond what past life expectancy alone provides (and vice versa). We use the SSR-based F-test with maxlag=7 years, then apply Bonferroni correction across all 29 country-tests.

**Key assumption:** Stationarity. We apply first-differences when the ADF test fails to reject a unit root.

**Interpretation:** Granger causality tests statistical *predictability* (temporal precedence), not structural causality. It is a necessary but not sufficient condition for causality.

In [ ]:
g = results['granger']
print('GDP → LE: significant in', g['n_sig_gdp2le'], '/', g['n_countries'], 'countries',
      f"({g['summary']['pct_sig_gdp2le']}%) — after Bonferroni correction")
print('LE → GDP: significant in', g['n_sig_le2gdp'], '/', g['n_countries'], 'countries',
      f"({g['summary']['pct_sig_le2gdp']}%) — suggesting reverse causality")
print('Most common optimal lag (GDP→LE):', g['summary']['most_common_lag_gdp2le'], 'year(s)')

# Pooled Granger results
print('\nPooled panel Granger test (country-demeaned):') 
for lag, f in sorted(g['pooled_fstats_gdp2le'].items())[:5]:
    p = g['pooled_pvals_gdp2le'][lag]
    print(f'  Lag {lag}: F={f:.2f}, p={p:.4f}')

In [ ]:
display(Image(CAUSAL_FIG_DIR / '01_granger_heatmap.png'))
display(Image(CAUSAL_FIG_DIR / '02_granger_lag_dist.png'))

**Finding:** After Bonferroni correction, GDP Granger-causes life expectancy in only 6.9% of countries. Notably, life expectancy Granger-causes GDP in 17.2% of countries, suggesting stronger evidence for the **reverse causal channel** (healthier populations are more productive → higher GDP). The modal optimal lag is 1 year.

> *Caveat:* With only 25 time periods per country and conservative Bonferroni correction, the power to detect lagged effects is limited. The pooled panel test (using demeaned data) shows more evidence of temporal precedence.

---
## 2. Panel Fixed Effects (Two-Way FE)

**Model:**
$$\text{LifeExp}_{it} = \beta_1 \ln(\text{GDP/cap})_{it} + \mathbf{X}_{it}\boldsymbol{\gamma} + \alpha_i + \delta_t + \varepsilon_{it}$$

where $\alpha_i$ are country fixed effects (absorbing time-invariant differences) and $\delta_t$ are year fixed effects (absorbing global trends). $\mathbf{X}$ includes health spending, education spending, governance, urbanisation, fertility, trade openness, age structure, and inflation.

**SE:** Clustered by country (Liang-Zeger).

**Subgroups:** Run separately for high-, middle-, and low-income countries.

In [ ]:
for spec, r in results['panel_fe'].items():
    stars = '***' if r.pval_gdp < 0.01 else '**' if r.pval_gdp < 0.05 else '*' if r.pval_gdp < 0.10 else ''
    print(f'[{spec}] β_GDP={r.coef_gdp:.3f}{stars} (se={r.se_gdp:.3f}, p={r.pval_gdp:.4f}) R²={r.rsquared:.3f} N={r.nobs}')

print('\n--- Subgroups ---')
for ig, r in results['panel_fe_subgroups'].items():
    stars = '***' if r.pval_gdp < 0.01 else '**' if r.pval_gdp < 0.05 else '*' if r.pval_gdp < 0.10 else ''
    print(f'[{ig}] β_GDP={r.coef_gdp:.3f}{stars} (se={r.se_gdp:.3f}, p={r.pval_gdp:.4f}) N={r.nobs}')

In [ ]:
display(Image(CAUSAL_FIG_DIR / '03_panel_coef_plot.png'))
display(Image(CAUSAL_FIG_DIR / '04_fe_within_variation.png'))

**Finding:** After controlling for country and year fixed effects, the GDP coefficient is **statistically insignificant** across all full-sample specifications. This is consistent with recent literature (Biggs et al. 2010; Gerdtham & Ruhm 2006) — within-country fluctuations in GDP have limited predictive power for within-country life expectancy changes.

However, the **high-income subgroup** yields a significant positive coefficient (β ≈ 5.0, p < 0.05), suggesting that for wealthy nations, GDP cycles do translate to health outcomes — possibly through responsive healthcare systems.

---
## 3. Instrumental Variable (2SLS)

**Motivation:** GDP and life expectancy are simultaneously determined. Healthier workers are more productive → higher GDP (reverse causality). OLS and TWFE estimates are biased if this channel is important.

**Instruments:**
1. **External demand shock** (Bartik-style): For country $i$, year $t$: the GDP-weighted mean GDP growth of all OTHER countries. This captures global demand shocks that affect a country's income through exports, but are unlikely to directly affect domestic health outcomes (exclusion restriction).
2. **Trade-exposure × external demand** (interaction): Countries more open to trade experience global demand shocks more strongly.

**Validity checks:** First-stage F-statistic (should exceed 10 to avoid weak instruments; Stock-Yogo); Sargan over-identification test (should not reject).

In [ ]:
for spec, r in results['iv'].items():
    stars = '***' if r.pval_gdp < 0.01 else '**' if r.pval_gdp < 0.05 else '*' if r.pval_gdp < 0.10 else ''
    print(f'[{spec}]')
    print(f'  β_GDP = {r.coef_gdp:.3f}{stars} (se={r.se_gdp:.3f}, p={r.pval_gdp:.4f})')
    print(f'  1st-stage F = {r.first_stage_fstat:.1f} (threshold: 10) — weak={r.weak_instrument}')
    print(f'  Sargan over-id p = {r.sargan_pval:.3f}' if r.sargan_pval else '  Sargan: just-identified')
    print(f'  Wu-Hausman endogeneity p = {r.wu_hausman_pval:.3f}' if r.wu_hausman_pval else '')

In [ ]:
display(Image(CAUSAL_FIG_DIR / '05_iv_first_stage.png'))
display(Image(CAUSAL_FIG_DIR / '06_iv_vs_ols.png'))

**Finding:** IV estimates are **large and strongly significant** (β ≈ 8.1–8.7 years per log-unit of GDP), with strong first-stage F-statistics (F ≈ 102–210, well above the 10 threshold) and a passing Sargan test (p = 0.46). The IV estimate substantially exceeds the OLS/TWFE estimate, which is consistent with the **local average treatment effect (LATE)** interpretation: the instrument identifies the effect for countries most responsive to global demand shocks, where the income–health link may be stronger.

The Wu-Hausman test p ≈ 0.09–0.11 provides weak evidence of endogeneity, suggesting the OLS/FE estimates are not severely biased but the IV LATE may capture a different (larger) margin of the GDP–health relationship.

---
## 4. Difference-in-Differences (Event Studies)

**Design:** We exploit three distinct universal health coverage (UHC) reform events:
- **Indonesia (JKN, 2014):** Launch of Jaminan Kesehatan Nasional (national health insurance), covering 90%+ of the population by 2019.
- **Vietnam (~2009):** Major push to universal health insurance coverage under the Health Insurance Law.
- **China (NCMS, 2009):** New Rural Cooperative Medical Scheme reached near-universal rural coverage.

**Identification:** Parallel pre-trends assumption — treated and control countries share common life expectancy trends before the reform.

**Event study:** We interact treatment × relative-year indicators to trace the dynamic treatment effect over time.

In [ ]:
for event_name, r in results['did'].items():
    stars = '***' if r.pval_att < 0.01 else '**' if r.pval_att < 0.05 else '*' if r.pval_att < 0.10 else ''
    pt = f'{r.parallel_trends_pval:.3f}' if r.parallel_trends_pval else 'N/A'
    print(f'[{r.label}]')
    print(f'  ATT = {r.att:.3f}{stars} (se={r.se_att:.3f}, p={r.pval_att:.4f})')
    print(f'  95% CI = [{r.ci95_att[0]:.3f}, {r.ci95_att[1]:.3f}]')
    print(f'  Parallel trends p = {pt}  |  N = {r.nobs}')

In [ ]:
for name in ['idn-jkn-2014', 'vnm-uhc-2009', 'chn-ncms-2009']:
    display(Image(CAUSAL_FIG_DIR / f'07_did_trends_{name}.png'))
    display(Image(CAUSAL_FIG_DIR / f'08_event_study_{name}.png'))

**Findings:**
- **Indonesia JKN (2014):** Significant positive ATT = +0.54 years (p = 0.04). Universal health coverage appears to have raised life expectancy by approximately half a year relative to the synthetic control group.
- **Vietnam UHC (2009):** Significant *negative* ATT = −0.97 years (p = 0.001). This counterintuitive result requires caution — it likely reflects a violation of the parallel trends assumption: Vietnam's control countries (Indonesia, Philippines, Egypt) experienced faster life-expectancy growth in the post-period for independent reasons.
- **China NCMS (2009):** ATT = −0.64 years, not significant (p = 0.48). Difficult to identify cleanly given China's many simultaneous policy changes.

> *Limitation:* DiD with only 4–5 countries in treated + control groups provides limited statistical power and makes parallel-trends testing underpowered. Results should be interpreted directionally.

---
## 5. Synthetic Control (China NCMS, 2009)

**Method:** We construct a *synthetic China* as a convex combination of the 28 other countries that best matches China's pre-2009 life expectancy trajectory. The post-2009 gap between actual and synthetic China is the estimated causal effect of the health reforms.

**Validity:** Pre-period RMSPE should be close to zero (good pre-period fit). Permutation p-value from placebo tests (apply same method to each donor and compare RMSPE ratios).

In [ ]:
s = results['synth']
print(f'Pre-period RMSPE:  {s.pre_rmspe:.3f} years (excellent if < 1.0)')
print(f'Post-period ATT:   {s.post_att:.3f} years')
print(f'Permutation p:     {s.p_value}')
print()
top_w = sorted(s.weights.items(), key=lambda x: -x[1])[:5]
print('Donor weights (top 5):')
for d, w in top_w:
    print(f'  {d}: {w:.3f}')

In [ ]:
display(Image(CAUSAL_FIG_DIR / '09_synthetic_control.png'))

**Finding:** The synthetic control achieves an excellent pre-period fit (RMSPE = 0.026 years — essentially perfect). The estimated post-2009 ATT is **+0.87 years** (China's actual life expectancy is 0.87 years higher than the synthetic counterfactual). The permutation p-value is 0.25 — not significant at conventional levels, but directionally consistent with a positive health reform effect.

---
## 6. Robustness Checks

In [ ]:
for spec_name, spec_res in results['robustness'].items():
    if spec_name == 'pooled_ols':
        r = spec_res
        print(f'[{spec_name}] β_GDP={r["coef_gdp"]:.3f} se={r["se_gdp"]:.3f} p={r["pval_gdp"]:.4f} N={r["nobs"]}')
    else:
        r_full = spec_res.get('controls_full') or spec_res.get('controls_base')
        if r_full:
            print(f'[{spec_name}] β_GDP={r_full.coef_gdp:.3f} p={r_full.pval_gdp:.4f} N={r_full.nobs}')

In [ ]:
display(Image(CAUSAL_FIG_DIR / '10_robustness.png'))

---
## 7. Triangulation — Synthesis of All Methods

In [ ]:
print(results['synthesis']['coef_table'].to_string(index=False))
display(Image(CAUSAL_FIG_DIR / '11_synthesis_forest.png'))
display(Image(CAUSAL_FIG_DIR / '12_mechanism_paths.png'))

## Summary of Causal Findings

| Evidence source | GDP → LE effect | Significance | Notes |
|---|---|---|---|
| Granger causality | Weak temporal precedence | 6.9% countries | Bonferroni-corrected; short series |
| Panel FE (TWFE) | Near-zero within-country | Not significant | High-income subgroup: β≈5.0, p<0.05 |
| IV-2SLS | +8.1–8.7 yrs / log-unit | p < 0.001 | Strong instruments (F>100); Sargan passed |
| DiD (IDN health reform) | +0.54 yrs | p = 0.04 | UHC expansion, 2014 |
| DiD (VNM) | −0.97 yrs | p = 0.001 | Likely parallel-trends violation |
| Synthetic control | +0.87 yrs | p = 0.25 | China NCMS 2009; non-significant |

### Key conclusions
1. **The GDP–life expectancy relationship is primarily between-country, not within-country.** Year-to-year GDP fluctuations within countries do not consistently translate to life expectancy changes — the cross-sectional Preston curve relationship (more income → longer lives) is robust, but the causal within-country channel is weaker.
2. **IV estimates suggest a substantial LATE:** when income increases are driven by global trade demand, the health returns are large (≈8 years per log-unit). This likely captures a specific margin (export-oriented economies, trade-responsive health infrastructure).
3. **Health system reforms matter independently:** The Indonesia JKN result suggests that Universal Health Coverage can produce significant gains in life expectancy (+0.5 years) even controlling for GDP trends.
4. **Mechanisms:** The GDP → health spending → life expectancy channel appears to be the dominant pathway (see mechanism plot). Governance quality moderates this link.

---
## Generate all outputs

In [ ]:
# Re-generate all figures and tables
fig_paths = run_all_causal_plots(results, df)
tbl_paths = run_all_tables(results)
print(f'Generated {len(fig_paths)} figures and {len(tbl_paths)} LaTeX tables.')